<a href="https://colab.research.google.com/github/shahad030/my-Misk-data-analysis-tasks/blob/main/stc_TV_T3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STC Jawwy

In [3]:
"""
Here we install libraries that are not installed by default
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [4]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices

# Jawwy dataset
The dataset consists of details about each customer and the movies and/or tv shows watched in addition to the genre.

You are required to work on task three to build a recommendation engine for our platform to Recommend movies to usesrs that they might be interested in¶


In [5]:
df = pd.read_excel("stc- Jawwy TV Data Set_T3.xlsb",index_col=0)
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [6]:
# check the data shape
df.shape

(1048575, 5)

In [7]:
# display the first 5 rows
df.head()

,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,42882,Drama
1,7946,Moana,1,42876,Animation
2,7418,The Mermaid Princess,1,42957,Animation
3,19307,The Mermaid Princess,2,42942,Animation
4,15860,Churchill,2,42923,Biography


In [8]:
# describe the numeric values in the dataset
df.describe()

,user_id_maped,rating,date_
count,1.048575e+06,1.048575e+06,1.048575e+06
mean,1.709266e+04,2.497283e+00,4.301202e+04
std,1.003513e+04,1.119837e+00,1.242834e+02
min,1.000000e+00,1.000000e+00,4.280800e+04
25%,8.253000e+03,1.000000e+00,4.289600e+04
50%,1.714900e+04,2.000000e+00,4.302200e+04
75%,2.566500e+04,3.000000e+00,4.312100e+04
max,3.428000e+04,4.000000e+00,4.322000e+04


In [9]:
# check if any column has null value in the dataset
df.isnull().any()

,0
user_id_maped,False
program_name,False
rating,False
date_,False
program_genre,False


In [10]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [11]:
"""
TODO build your Recommender system to Highlight Programs that usesrs might be interested in
"""
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
import warnings, time
warnings.filterwarnings("ignore")

t0 = time.time()


df.columns = df.columns.str.strip()
df.reset_index(drop=True, inplace=True)

print(f" Loaded in {time.time()-t0:.1f}s")
print(f"   Rows: {len(df):,}  |  Users: {df['user_id_maped'].nunique():,}  |  Programs: {df['program_name'].nunique():,}")
print()

# ─────────────────────────────────────────────
# 2. ENCODE USERS & PROGRAMS AS INTEGER INDICES
#    (much faster than string lookups)
# ─────────────────────────────────────────────
df["liked"] = (df["rating"] >= 3).astype(np.int8)

users   = df["user_id_maped"].unique()
programs = df["program_name"].unique()

user2idx = {u: i for i, u in enumerate(users)}
prog2idx = {p: i for i, p in enumerate(programs)}
idx2prog = {i: p for p, i in prog2idx.items()}

df["u_idx"] = df["user_id_maped"].map(user2idx)
df["p_idx"] = df["program_name"].map(prog2idx)

n_users = len(users)
n_progs = len(programs)

# ─────────────────────────────────────────────
# 3. BUILD SPARSE USER-ITEM RATING MATRIX
#    shape: (n_users, n_programs)
# ─────────────────────────────────────────────
# Average rating if a user watched the same program multiple times
agg = df.groupby(["u_idx","p_idx"])["rating"].mean().reset_index()

user_item_sparse = csr_matrix(
    (agg["rating"].values, (agg["u_idx"].values, agg["p_idx"].values)),
    shape=(n_users, n_progs)
)

# Boolean mask: True where user HAS watched the program
watched_mask = (user_item_sparse > 0).toarray()   # shape (n_users, n_progs)

# ─────────────────────────────────────────────
# 4. COLLABORATIVE FILTERING — SVD
#    Fit on sparse matrix, reconstruct dense scores
# ─────────────────────────────────────────────
t1 = time.time()
print("⏳ Running SVD (Collaborative Filtering)...")

n_components = min(20, n_users - 1, n_progs - 1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
U  = svd.fit_transform(user_item_sparse)   # (n_users, k)
Vt = svd.components_                        # (k, n_progs)

# Full predicted-rating matrix for ALL users at once
cf_scores = np.dot(U, Vt)                  # (n_users, n_progs)

print(f" SVD done in {time.time()-t1:.1f}s")

# ─────────────────────────────────────────────
# 5. CONTENT-BASED FILTERING — Genre TF-IDF
#    Build a (n_users, n_progs) content score matrix
#    by summing cosine similarities weighted by user's liked ratings
# ─────────────────────────────────────────────
t2 = time.time()
print("⏳ Building Content-Based scores...")

# One row per program, genre as text
prog_meta = (
    df.drop_duplicates("p_idx")[["p_idx","program_name","program_genre"]]
    .set_index("p_idx")
    .sort_index()
)

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(prog_meta["program_genre"])        # (n_progs, vocab)
prog_sim = cosine_similarity(genre_matrix, genre_matrix)              # (n_progs, n_progs)

# Liked-rating matrix: only keep ratings where liked=1, else 0
liked_agg = df[df["liked"] == 1].groupby(["u_idx","p_idx"])["rating"].mean().reset_index()
liked_sparse = csr_matrix(
    (liked_agg["rating"].values, (liked_agg["u_idx"].values, liked_agg["p_idx"].values)),
    shape=(n_users, n_progs)
)

# Content score = liked_ratings @ prog_sim
# Shape: (n_users, n_progs) — one vectorized matrix multiply, no loops!
cb_scores = liked_sparse.dot(prog_sim)     # (n_users, n_progs)

# Normalize to [0,1] range so scales match CF
def minmax(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

cf_norm = minmax(cf_scores)
cb_norm = minmax(cb_scores)

print(f" Content-Based done in {time.time()-t2:.1f}s")

# ─────────────────────────────────────────────
# 6. HYBRID SCORE MATRIX
#    60% collaborative + 40% content-based
#    Then zero-out already-watched programs
# ─────────────────────────────────────────────
hybrid = 0.6 * cf_norm + 0.4 * cb_norm     # (n_users, n_progs)
hybrid[watched_mask] = -np.inf              # mask out already-watched — no loops needed

# ─────────────────────────────────────────────
# 7. EXTRACT TOP-5 FOR EVERY USER (vectorized)
# ─────────────────────────────────────────────
TOP_N = 5
# argsort descending — get top-N column indices per row
top5_idx = np.argsort(-hybrid, axis=1)[:, :TOP_N]   # (n_users, 5)

# Map back to program names
top5_names = np.vectorize(idx2prog.get)(top5_idx)    # (n_users, 5)

# ─────────────────────────────────────────────
# 8. TASK 1 — Program highlights for ALL users
# ─────────────────────────────────────────────
print()
print("=" * 60)
print(" TASK 1: PERSONALIZED HIGHLIGHTS — ALL USERS")
print("=" * 60)

highlights_df = pd.DataFrame(
    top5_names,
    index=users,
    columns=[f"Rec #{i+1}" for i in range(TOP_N)]
)
highlights_df.index.name = "user_id_maped"
print(highlights_df.to_string())

Streaming output truncated to the last 5000 lines.
28866                                                                      The Boss Baby                                                                     Trolls                                                                      Moana                                                       The Mermaid Princess                                                                   Baywatch
31798                                                                      The Boss Baby                                                                     Trolls                                                                      Moana                                                       The Mermaid Princess                                                      Surf's Up : WaveMania
24193                                                                      The Boss Baby                                                                     Trolls        

In [12]:
"""
TODO show the recommendations (top 5) for the people who watched "Moana" movie
"""
print()
print("=" * 60)
print("🎬 TASK 2: TOP 5 FOR USERS WHO WATCHED 'Moana'")
print("=" * 60)

moana_user_ids  = df[df["program_name"] == "Moana"]["user_id_maped"].unique()
moana_u_indices = [user2idx[u] for u in moana_user_ids if u in user2idx]

moana_recs = {}
for uid, uidx in zip(moana_user_ids, moana_u_indices):
    recs = top5_names[uidx].tolist()
    moana_recs[uid] = recs
    moana_rating = df[(df["user_id_maped"] == uid) & (df["program_name"] == "Moana")]["rating"].values[0]
    emoji = "❤️" if moana_rating >= 3 else "👀"
    print(f"\nUser {uid} {emoji}  (rated Moana: {moana_rating}/4)")
    for i, rec in enumerate(recs, 1):
        genre = prog_meta[prog_meta["program_name"] == rec]["program_genre"].values
        genre_str = genre[0] if len(genre) > 0 else "Unknown"
        print(f"   {i}. {rec:<45} [{genre_str}]")

moana_recs_df = pd.DataFrame.from_dict(
    moana_recs, orient="index",
    columns=[f"Top {i+1}" for i in range(TOP_N)]
)
moana_recs_df.index.name = "user_id_maped"

# Aggregate: most recommended across all Moana watchers
all_flat = [r for recs in moana_recs.values() for r in recs]
top_overall = pd.Series(all_flat).value_counts().head(TOP_N)

print()
print("=" * 60)
print("🏆 OVERALL TOP 5 — Most recommended across ALL Moana watchers")
print("=" * 60)
for rank, (prog, count) in enumerate(top_overall.items(), 1):
    genre = prog_meta[prog_meta["program_name"] == prog]["program_genre"].values
    genre_str = genre[0] if len(genre) > 0 else "Unknown"
    print(f"  #{rank}  {prog:<45} [{genre_str}]  — recommended to {count} users")

# ─────────────────────────────────────────────
# 10. SAVE OUTPUTS
# ─────────────────────────────────────────────
print()
highlights_df.to_csv("task1_user_highlights.csv")
moana_recs_df.to_csv("task2_moana_recommendations.csv")
top_overall.rename_axis("program_name").reset_index(name="recommended_to_n_users").to_csv(
    "task2_overall_top5.csv", index=False)

print(f"✅ All done in {time.time()-t0:.1f}s total")
print("   → task1_user_highlights.csv")
print("   → task2_moana_recommendations.csv")
print("   → task2_overall_top5.csv")

Streaming output truncated to the last 5000 lines.
   4. The Adventures of Petey and Friends           [Animation]
   5. The BFG                                       [Family]

User 25381 👀  (rated Moana: 2/4)
   1. The Boss Baby                                 [Animation]
   2. The Mermaid Princess                          [Animation]
   3. Surf's Up : WaveMania                         [Animation]
   4. The Jetsons & WWE: Robo-WrestleMania!         [Animation]
   5. Rings                                         [Horror]

User 13109 👀  (rated Moana: 2/4)
   1. Vikings     The Plan                          [Action]
   2. Vikings     The Departed  Part                [Action]
   3. Vikings     Homeland                          [Action]
   4. Vikings     The Message                       [Action]
   5. Vikings     The Prisoner                      [Action]

User 19465 👀  (rated Moana: 2/4)
   1. The Boss Baby                                 [Animation]
   2. Trolls                        